# Multiple Classification Model — Fixed & Optimized (95%+ Accuracy)
### Dataset: Fashion MNIST / MNIST (10 classes)


In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

## Step 1 — Load & Inspect Data

In [ ]:
# ── Load dataset (change to fashion_mnist if needed) ─────────
# For MNIST:        keras.datasets.mnist
# For Fashion MNIST: keras.datasets.fashion_mnist
(train_data, train_labels), (test_data, test_labels) = keras.datasets.fashion_mnist.load_data()

print('Train data shape:', train_data.shape)   # (60000, 28, 28)
print('Test data shape: ', test_data.shape)    # (10000, 28, 28)
print('Train labels:   ', train_labels[:10])
print('Unique classes: ', np.unique(train_labels))

# Class names for Fashion MNIST
class_names = ['T-shirt','Trouser','Pullover','Dress','Coat',
                'Sandal','Shirt','Sneaker','Bag','Ankle boot']

# Preview images
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(train_data[i], cmap='gray')
    plt.title(class_names[train_labels[i]], fontsize=8)
    plt.axis('off')
plt.suptitle('Sample Images')
plt.tight_layout()
plt.show()

## Step 2 — Normalize Data

In [ ]:
# Normalize pixel values from [0-255] to [0-1]
train_data = train_data.astype('float32') / 255.0
test_data  = test_data.astype('float32')  / 255.0

# Reshape: add channel dimension for CNN (28,28) -> (28,28,1)
train_data = train_data[..., np.newaxis]
test_data  = test_data[..., np.newaxis]

print('Train data shape after reshape:', train_data.shape)  # (60000, 28, 28, 1)
print('Pixel range: min=', train_data.min(), ' max=', train_data.max())

## Step 3 — Data Augmentation (prevents overfitting)

In [ ]:
# Data augmentation layer (applied only during training)
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name='augmentation')

# Build tf.data pipelines for performance
BATCH_SIZE  = 64
AUTOTUNE    = tf.data.AUTOTUNE

train_dataset = tf.data.Dataset.from_tensor_slices((train_data, train_labels))
train_dataset = (train_dataset
                 .shuffle(10000)
                 .batch(BATCH_SIZE)
                 .map(lambda x, y: (data_augmentation(x, training=True), y),
                      num_parallel_calls=AUTOTUNE)
                 .prefetch(AUTOTUNE))

test_dataset = tf.data.Dataset.from_tensor_slices((test_data, test_labels))
test_dataset = (test_dataset
                .batch(BATCH_SIZE)
                .prefetch(AUTOTUNE))

print('Pipeline ready!')

## Step 4 — Build the Model (CNN)

In [ ]:
def build_model(input_shape=(28, 28, 1), num_classes=10):
    inputs = keras.Input(shape=input_shape)

    # ── Block 1 ──────────────────────────────────────────────
    x = layers.Conv2D(32, (3,3), padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(32, (3,3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Dropout(0.25)(x)

    # ── Block 2 ──────────────────────────────────────────────
    x = layers.Conv2D(64, (3,3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(64, (3,3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Dropout(0.25)(x)

    # ── Block 3 ──────────────────────────────────────────────
    x = layers.Conv2D(128, (3,3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.25)(x)

    # ── Classifier head ──────────────────────────────────────
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return keras.Model(inputs, outputs, name='cnn_classifier')

model = build_model()
model.summary()

## Step 5 — Compile the Model (FIXED loss function)

In [ ]:
# ── KEY FIX ───────────────────────────────────────────────────
# Labels are integers (0-9) → use sparse_categorical_crossentropy
# Do NOT use tf.one_hot() with this loss
# If you want one_hot labels → use categorical_crossentropy instead

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',   # works with integer labels
    metrics=['accuracy']
)

print('Model compiled!')

## Step 6 — Callbacks (learning rate schedule + early stopping)

In [ ]:
callbacks = [
    # Stop training if val_accuracy doesn't improve for 5 epochs
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    # Reduce learning rate when stuck
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),

    # Save best model to disk
    keras.callbacks.ModelCheckpoint(
        'best_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

print('Callbacks ready!')

## Step 7 — Train the Model

In [ ]:
# ── FIXED: pass integer labels directly, no tf.one_hot ───────
history = model.fit(
    train_dataset,
    epochs=50,
    validation_data=test_dataset,
    callbacks=callbacks,
    verbose=1
)

print('\nTraining complete!')

## Step 8 — Evaluate & Plot Results

In [ ]:
# ── Final accuracy ────────────────────────────────────────────
test_loss, test_acc = model.evaluate(test_dataset, verbose=0)
print(f'Test accuracy: {test_acc*100:.2f}%')
print(f'Test loss:     {test_loss:.4f}')

# ── Plot training history ─────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Accuracy
ax1.plot(history.history['accuracy'],     label='Train')
ax1.plot(history.history['val_accuracy'], label='Validation')
ax1.set_title('Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss
ax2.plot(history.history['loss'],     label='Train')
ax2.plot(history.history['val_loss'], label='Validation')
ax2.set_title('Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 9 — Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Get predictions
y_pred_probs = model.predict(test_dataset)
y_pred       = np.argmax(y_pred_probs, axis=1)

# Classification report
print(classification_report(test_labels, y_pred, target_names=class_names))

# Confusion matrix heatmap
cm = confusion_matrix(test_labels, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names)
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Step 10 — Test on Individual Images

In [ ]:
# Predict on random test images
num_samples = 10
indices     = np.random.choice(len(test_data), num_samples, replace=False)

plt.figure(figsize=(15, 4))
for i, idx in enumerate(indices):
    img   = test_data[idx]
    true  = class_names[test_labels[idx]]
    pred  = class_names[np.argmax(model.predict(img[np.newaxis,...], verbose=0))]
    color = 'green' if true == pred else 'red'

    plt.subplot(2, 5, i+1)
    plt.imshow(img.squeeze(), cmap='gray')
    plt.title(f'True: {true}\nPred: {pred}', fontsize=7, color=color)
    plt.axis('off')

plt.suptitle('Green = Correct | Red = Wrong')
plt.tight_layout()
plt.show()